# Ejercicios de Structured Streaming: Modos de Salida y Triggers

Este cuaderno desarrolla las pruebas prácticas de comparación entre los modos de salida `complete`, `update` y `append`, así como la configuración de intervalos de disparo mediante `trigger`.

In [1]:
import os
import sys
import socket
import threading
import time

jdk_path = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.10002-hotspot"
if os.path.exists(jdk_path):
    os.environ["JAVA_HOME"] = jdk_path
    os.environ["PATH"] = jdk_path + r"\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession \
    .builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

def enviar_frases_socket(puerto, espera_inicial=2, pausa_entre_frases=3):
    frases = [
        "Esta es la primera frase",
        "Enviamos segunda linea primera"
    ]
    servidor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    servidor.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    servidor.bind(("localhost", puerto))
    servidor.listen(1)
    conn, _ = servidor.accept()
    time.sleep(espera_inicial)
    for frase in frases:
        conn.sendall((frase + "\n").encode("utf-8"))
        time.sleep(pausa_entre_frases)
    time.sleep(3)
    conn.close()
    servidor.close()

def crear_flujo(puerto):
    lineasDF = spark.readStream \
        .format("socket") \
        .option("host", "localhost") \
        .option("port", str(puerto)) \
        .load()
    palabrasDF = lineasDF.select(explode(split(lineasDF.value, ' ')).alias('palabra'))
    return palabrasDF.groupBy("palabra").count()

## Ejercicio 1: Comparación de los modos de salida

### 1. Modo `complete`

Al enviar primero `Esta es la primera frase` y luego `Enviamos segunda linea primera`, el modo `complete` muestra en cada micro-lote la tabla íntegra con todas las palabras acumuladas desde el inicio de la consulta.

In [2]:
threading.Thread(target=enviar_frases_socket, args=(9999, 2, 3), daemon=True).start()
cantidadDF_complete = crear_flujo(9999)

wordCountComplete = cantidadDF_complete.writeStream \
    .format("console") \
    .outputMode("complete") \
    .start()

wordCountComplete.awaitTermination(timeout=10)
wordCountComplete.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
|     es|    1|
|  frase|    1|
|   Esta|    1|
|     la|    1|
|primera|    1|
+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|   linea|    1|
|      es|    1|
|Enviamos|    1|
| segunda|    1|
|      la|    1|
| primera|    2|
|   frase|    1|
|    Esta|    1|
+--------+-----+


### 2. Modo `update`

Al cambiar el modo de salida a `update`, reiniciar la consulta y enviar las mismas dos frases, cada micro-lote muestra únicamente los registros que fueron insertados o cuyo conteo cambió en ese lote específico. En el segundo lote solo aparecen `Enviamos`, `segunda`, `linea` con conteo 1 y `primera` actualizado a 2, omitiendo las palabras de la primera frase que no sufrieron cambios.

In [3]:
threading.Thread(target=enviar_frases_socket, args=(9998, 2, 3), daemon=True).start()
cantidadDF_update = crear_flujo(9998)

wordCountUpdate = cantidadDF_update.writeStream \
    .format("console") \
    .outputMode("update") \
    .start()

wordCountUpdate.awaitTermination(timeout=10)
wordCountUpdate.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
|     es|    1|
|   Esta|    1|
|     la|    1|
|primera|    1|
|  frase|    1|
+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|   linea|    1|
|Enviamos|    1|
| segunda|    1|
| primera|    2|
+--------+-----+


### 3. Modo `append`

Al configurar el modo de salida en `append` sobre un DataFrame que contiene una agregación sin marca de agua o watermark, Spark impide iniciar la consulta y lanza la excepción `AnalysisException`. Esto sucede porque el modo `append` exige que las filas emitidas sean definitivas e inmutables, pero en un conteo continuo cualquier palabra puede volver a aparecer en el futuro y modificar su total acumulado.

In [4]:
try:
    wordCountAppend = cantidadDF_update.writeStream \
        .format("console") \
        .outputMode("append") \
        .start()
except Exception as e:
    print(f"{type(e).__name__}: {str(e).splitlines()[0]}")

AnalysisException: Append output mode not supported when there are streaming aggregations on streaming DataFrames/DataSets without watermark;


## Ejercicio 2: Triggers

Al añadir `.trigger(processingTime="1 minute")`, Spark agrupa todos los mensajes recibidos durante esa ventana de un minuto en un único micro-lote. Si se envían las dos frases con pocos segundos de diferencia, la consola no imprime el resultado de inmediato tras cada línea, sino que espera hasta que se cumple el intervalo del disparador y procesa ambas frases juntas en un solo lote.

In [5]:
threading.Thread(target=enviar_frases_socket, args=(9997, 1, 1), daemon=True).start()
cantidadDF_trigger = crear_flujo(9997)

wordCountTrigger = cantidadDF_trigger.writeStream \
    .format("console") \
    .outputMode("complete") \
    .trigger(processingTime="1 minute") \
    .start()

wordCountTrigger.awaitTermination(timeout=65)
wordCountTrigger.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|   linea|    1|
|      es|    1|
|Enviamos|    1|
| segunda|    1|
|      la|    1|
| primera|    2|
|   frase|    1|
|    Esta|    1|
+--------+-----+
